# El ganso del Nilo en Europa

<div style="padding: 14px 18px; border-left: 4px solid d95f3d; background: #f7f4ee; color: #24343b;">
<strong>Una exploración visual de la expansión de <em>Alopochen aegyptiaca</em></strong><br>
Observaciones con coordenadas publicadas en GBIF, desde 1970 hasta 2025. El recorrido combina conteos anuales, un mapa de registros y una animación.
</div>

> **Fuente:** [GBIF](https://www.gbif.org/) · Los conteos representan las ocurrencias que cumplen los filtros; la descarga cartográfica limita el número de registros por año para mantener el proceso manejable.

## 1. Preparar el análisis

El notebook importa el script original para conservar sus consultas, validaciones y caché. Si vuelves a ejecutarlo, los datos ya descargados se reutilizan.

In [ ]:
from IPython.display import HTML, Image, display
import matplotlib.pyplot as plt
import pandas as pd
import requests

import ganso_europa as ge

plt.rcParams.update({
    "figure.facecolor": "white",
    "axes.facecolor": "white",
    "axes.titlecolor": "#24343b",
    "axes.labelcolor": "#46545a",
    "xtick.color": "#46545a",
    "ytick.color": "#46545a",
    "font.size": 11,
})

display(HTML(f"""
<div style="display:flex; gap:12px; flex-wrap:wrap; margin:10px 0 18px">
  <div style="padding:12px 18px; background:#f7f4ee; border-radius:8px"><b>Especie</b><br><i>{ge.SPECIES_NAME}</i></div>
  <div style="padding:12px 18px; background:#f7f4ee; border-radius:8px"><b>Periodo</b><br>{ge.YEAR_MIN}–{ge.YEAR_MAX}</div>
  <div style="padding:12px 18px; background:#f7f4ee; border-radius:8px"><b>Muestra máxima</b><br>{ge.MAX_PER_YEAR:,} registros por año</div>
</div>
"""))

## 2. Consultar GBIF y explorar los conteos

La faceta anual ofrece el total de ocurrencias por año. La descarga de coordenadas que usaremos en los mapas está limitada por `MAX_PER_YEAR`.

In [ ]:
with requests.Session() as session:
    taxon_key = ge.get_taxon_key(session)
    year_counts, total_records = ge.get_year_counts(session, taxon_key)

display(HTML(
    f'<div style="padding:12px 16px; background:#edf3f2; border-radius:8px">'
    f'<b>TaxonKey:</b> {taxon_key} &nbsp; · &nbsp; '
    f'<b>Ocurrencias encontradas:</b> {total_records:,}</div>'
))

In [ ]:
years = pd.Series(year_counts, name="Ocurrencias").sort_index()
years = years.reindex(range(ge.YEAR_MIN, ge.YEAR_MAX + 1), fill_value=0)

fig, ax = plt.subplots(figsize=(12, 4.5))
ax.bar(years.index, years.values, color="#7296a5", width=0.82)
ax.set_title("Ocurrencias del ganso del Nilo por año", loc="left", fontsize=16, weight="bold", pad=14)
ax.set_ylabel("Registros en GBIF")
ax.set_xlabel("Año")
ax.set_xticks(range(ge.YEAR_MIN, ge.YEAR_MAX + 1, 5))
ax.grid(axis="y", color="#e5e8e7", linewidth=0.8)
ax.set_axisbelow(True)
for spine in ("top", "right", "left"):
    ax.spines[spine].set_visible(False)
ax.spines["bottom"].set_color("#d9dddc")
ax.tick_params(axis="y", length=0)
ax.margins(x=0.01)
fig.tight_layout()
plt.show()

## 3. Descargar y preparar las coordenadas

Los registros se proyectan a **EPSG:3035**, se filtran para el área del mapa y se eliminan coordenadas duplicadas. La primera ejecución puede tardar; las siguientes aprovechan los CSV guardados en `cache_gbif_ganso/`.

In [ ]:
with requests.Session() as session:
    points = ge.load_occurrences(session, taxon_key, year_counts)

if points.empty:
    raise RuntimeError("No quedaron registros con coordenadas dentro de la vista del mapa.")

display(HTML(
    f'<div style="padding:12px 16px; background:#edf3f2; border-radius:8px">'
    f'<b>Registros incluidos en los mapas:</b> {len(points):,} '
    f'&nbsp; · &nbsp; <b>Años con registros:</b> {points["year"].nunique()}</div>'
))

## 4. Mapa de observaciones

Cada punto representa una coordenada de la muestra descargada. El mapa ofrece una vista conjunta; la animación siguiente separa los registros por año.

In [ ]:
countries = ge.load_countries().to_crs("EPSG:3035")
fig, ax = plt.subplots(figsize=(10, 8), dpi=110)
countries.plot(ax=ax, color="#f1eee7", edgecolor="#c7c3bb", linewidth=0.45)
ax.scatter(points["x"], points["y"], s=7, color="#d95f3d", alpha=0.42, linewidths=0)
ax.set_xlim(*ge.MAP_X_LIMITS)
ax.set_ylim(*ge.MAP_Y_LIMITS)
ax.set_axis_off()
ax.set_title("Registros con coordenadas en Europa", loc="left", fontsize=16, weight="bold", pad=14)
fig.text(0.5, 0.035, "Fuente: GBIF · Muestra limitada por año · Proyección EPSG:3035", ha="center", color="#687276", fontsize=9)
fig.tight_layout(rect=(0, 0.05, 1, 1))
plt.show()

## 5. Animación anual

Se genera el mismo GIF que produce el script original y se muestra aquí mismo. El archivo queda guardado como `ganso_nilo_europa.gif`.

In [ ]:
ge.create_animation(points)
display(Image(filename=str(ge.OUTPUT_GIF)))